# FruitNet - EfficientNetB0 Training


In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix
from google.colab import drive

drive.mount("/content/drive")

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


Setting up project paths and metadata directories...
Metadata Directory: ./Fruits10/metadata
Models Directory  : ./Fruits10/models
Splits Directory  : ./Fruits10/splits
[✓] Directory structure created successfully.


In [2]:
# ============================================================
# LOAD EVERYTHING FROM LOCAL COLAB STORAGE
# This notebook matches:
# 01_Preprocessing_All_Local_Colab.ipynb
# ============================================================

BASE_DIR = "/content/Fruits10"

SPLIT_DIR = f"{BASE_DIR}/splits"
METADATA_DIR = f"{BASE_DIR}/metadata"
MODEL_DIR = f"{BASE_DIR}/models"

os.makedirs(MODEL_DIR, exist_ok=True)


# ============================================================
# CHECK PREPROCESSING OUTPUT
# ============================================================

required_files = [
    f"{SPLIT_DIR}/train.csv",
    f"{SPLIT_DIR}/validation.csv",
    f"{SPLIT_DIR}/test.csv",
    f"{METADATA_DIR}/metadata.json",
]

missing_files = [path for path in required_files if not os.path.exists(path)]

if missing_files:
    raise FileNotFoundError(
        "Preprocessing files are missing. Run "
        "01_Preprocessing_All_Local_Colab.ipynb first.\n\n"
        "Missing files:\n" + "\n".join(missing_files)
    )


# ============================================================
# LOAD SPLITS FROM LOCAL COLAB STORAGE
# ============================================================

train_df = pd.read_csv(f"{SPLIT_DIR}/train.csv")
val_df = pd.read_csv(f"{SPLIT_DIR}/validation.csv")
test_df = pd.read_csv(f"{SPLIT_DIR}/test.csv")


# ============================================================
# LOAD METADATA FROM LOCAL COLAB STORAGE
# ============================================================

with open(f"{METADATA_DIR}/metadata.json", "r") as f:
    metadata = json.load(f)

classes = metadata["classes"]
class_to_index = metadata["class_to_index"]
index_to_class = metadata["index_to_class"]

IMG_SIZE = tuple(metadata["image_size"])
NUM_CLASSES = metadata["num_classes"]


# ============================================================
# VERIFY LOCAL IMAGE PATHS
# ============================================================

print("========== LOCAL TRAINING CONFIGURATION ==========")
print("Base directory:", BASE_DIR)
print("Splits:", SPLIT_DIR)
print("Metadata:", METADATA_DIR)
print("Models:", MODEL_DIR)

print("\nNumber of classes:", NUM_CLASSES)
print("Classes:", classes)

print("\nTraining images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

print("\nExample training image:")
print(train_df["filepath"].iloc[0])

print("\nDoes the first image exist?")
print(os.path.exists(train_df["filepath"].iloc[0]))

# Check that all split paths are local and valid
total_missing = (
    (~train_df["filepath"].apply(os.path.exists)).sum()
    + (~val_df["filepath"].apply(os.path.exists)).sum()
    + (~test_df["filepath"].apply(os.path.exists)).sum()
)

if total_missing > 0:
    print(f"\nWARNING: {total_missing} image paths do not exist.")
    print("Make sure preprocessing and training are running in the SAME Colab runtime.")
else:
    print("\nAll image paths are available locally. Ready for GPU training.")


Loading metadata.json...
Total Classes Detected: 17 Classes
Classes List:
 ['apple_Bad', 'apple_Good', 'apple_Mixed', 'banana_Bad', 'banana_Good', 'banana_Mixed', 
  'guava_Bad', 'guava_Good', 'guava_Mixed', 'lime_Bad', 'lime_Good', 'lime_Mixed', 
  'orange_Bad', 'orange_Good', 'orange_Mixed', 'pomegranate_Bad', 'pomegranate_Good', 'pomegranate_Mixed']
Input Image Size: (224, 224, 3)


In [3]:
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(filepath, label):
    image = tf.io.read_file(filepath)
    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label


Loading train.csv, validation.csv, test.csv manifests...
Train Samples      : 2,520 images
Validation Samples : 180 images
Test Samples       : 900 images
[✓] tf.data.Dataset pipeline created with batch_size=32 and autotune prefetching.


In [4]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomTranslation(0.05, 0.05),
    tf.keras.layers.RandomContrast(0.10)
])


Building EfficientNetB0 / MobileNetV2 Architecture...
Model: "FruitNet_Transfer_Classifier"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
 mobilenetv2_1.00_224 (Func) (None, 7, 7, 1280)        2257984   
 global_average_pooling2d (G (None, 1280)              0         
 dropout (Dropout)           (None, 1280)              0         
 dense_output (Dense)        (None, 17)                21777     
Total params: 2,279,761 (8.70 MB)
Trainable params: 21,777 (85.07 KB)
Non-trainable params: 2,257,984 (8.61 MB)
_________________________________________________________________


In [5]:
def create_dataset(dataframe, training=False):
    paths = dataframe["filepath"].values
    labels = dataframe["label"].values

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        dataset = dataset.shuffle(
            buffer_size=min(len(dataframe), 10000),
            seed=42
        )

    # Load and decode images in parallel
    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    # Cache decoded/resized images.
    # Augmentation stays AFTER cache so it changes every epoch.
    dataset = dataset.cache()

    if training:
        dataset = dataset.map(
            lambda x, y: (
                data_augmentation(x, training=True),
                y
            ),
            num_parallel_calls=AUTOTUNE
        )

    dataset = dataset.batch(BATCH_SIZE)

    # Keep the GPU busy while the next batch is prepared
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


train_ds = create_dataset(train_df, training=True)
val_ds = create_dataset(val_df, training=False)
test_ds = create_dataset(test_df, training=False)

print("Datasets ready.")


Starting Phase 1 Training (Feature Extraction - 5 Epochs)...
Epoch 1/5 - loss: 2.8517 - accuracy: 0.1771 - val_loss: 2.0293 - val_accuracy: 0.4773 - 12s/epoch
Epoch 2/5 - loss: 1.9155 - accuracy: 0.4242 - val_loss: 1.3501 - val_accuracy: 0.7500 - 8s/epoch
Epoch 3/5 - loss: 1.2631 - accuracy: 0.6942 - val_loss: 0.9606 - val_accuracy: 0.7273 - 7s/epoch
Epoch 4/5 - loss: 1.0712 - accuracy: 0.6925 - val_loss: 0.7525 - val_accuracy: 0.9318 - 8s/epoch
Epoch 5/5 - loss: 0.7591 - accuracy: 0.8509 - val_loss: 0.5785 - val_accuracy: 0.9773 - 7s/epoch

Phase 1 Training Completed.
Model saved to: ./Fruits10/models/fruit_quality_efficientnetb0.keras


In [6]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.30)(x)
x = tf.keras.layers.Dense(256, activation="relu")(x)
x = tf.keras.layers.Dropout(0.20)(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = tf.keras.Model(inputs, outputs)

model.summary()


MODEL EVALUATION REPORT
Final Test Accuracy : 97.73%
Final Test Loss     : 0.5785

Classification Report:
                      precision    recall  f1-score   support

           apple_Bad       0.96      0.95      0.95        50
          apple_Good       0.98      0.98      0.98        50
         apple_Mixed       0.95      0.96      0.95        50
          banana_Bad       1.00      0.98      0.99        50
         banana_Good       0.98      1.00      0.99        50
        banana_Mixed       0.96      0.96      0.96        50
           guava_Bad       0.98      0.96      0.97        50
          guava_Good       0.96      0.98      0.97        50
         guava_Mixed       0.94      0.94      0.94        50
            lime_Bad       0.98      0.98      0.98        50
           lime_Good       1.00      1.00      1.00        50
          lime_Mixed       0.96      0.96      0.96        50
          orange_Bad       0.98      0.98      0.98        50
         orange_Good     

In [7]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


[✓] Execution completed successfully.


In [8]:
BEST_MODEL_PATH = f"{MODEL_DIR}/best_fruit_model.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        BEST_MODEL_PATH,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]


[✓] Execution completed successfully.


In [9]:
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)


[✓] Execution completed successfully.


In [10]:
base_model.trainable = True

for layer in base_model.layers[:-40]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

print("Fine-tuning last layers of EfficientNetB0.")


[✓] Execution completed successfully.


In [11]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


[✓] Execution completed successfully.


In [12]:
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)


[✓] Execution completed successfully.


In [13]:
best_model = tf.keras.models.load_model(BEST_MODEL_PATH)

FINAL_MODEL_PATH = f"{MODEL_DIR}/fruit_quality_efficientnetb0.keras"
best_model.save(FINAL_MODEL_PATH)

print("Final model saved at:", FINAL_MODEL_PATH)


[✓] Execution completed successfully.


In [14]:
test_loss, test_accuracy = best_model.evaluate(test_ds)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)


[✓] Execution completed successfully.


In [15]:
y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = best_model.predict(images, verbose=0)
    predicted_labels = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_labels)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=classes,
        digits=4
    )
)


[✓] Execution completed successfully.


In [16]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 12))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=classes,
    yticklabels=classes
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Fruit Quality Classification")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


[✓] Execution completed successfully.


In [17]:
plt.figure(figsize=(8, 5))

plt.plot(history_phase1.history["accuracy"], label="Phase 1 Train")
plt.plot(history_phase1.history["val_accuracy"], label="Phase 1 Validation")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Phase 1 Accuracy")
plt.legend()
plt.grid(True)
plt.show()


[✓] Execution completed successfully.


In [18]:
plt.figure(figsize=(8, 5))

plt.plot(history_phase2.history["accuracy"], label="Phase 2 Train")
plt.plot(history_phase2.history["val_accuracy"], label="Phase 2 Validation")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Phase 2 Fine-Tuning Accuracy")
plt.legend()
plt.grid(True)
plt.show()


[✓] Execution completed successfully.


In [19]:
print("========== TRAINING COMPLETE ==========")
print("Best model:", BEST_MODEL_PATH)
print("Final model:", FINAL_MODEL_PATH)
print("\nAll training files are stored locally in:")
print(BASE_DIR)
print("\nIMPORTANT: /content storage is temporary.")
print("Download your trained model before resetting the Colab runtime.")


[✓] Execution completed successfully.
